# Stage 10/11 — Pledged trend-following ranking

New: `score_for_pledged_trend_following()` in `strategy_scoring.py` — ranks the pledged-securities 1D universe for what it's actually for: buy the share/ETF itself on a fresh UP flip, exit on the opposite DOWN flip. Different from the other four scorers in this project — favors a large favorable move on the UP flips specifically (buying the underlying has no structural cap the way credit spreads/short IC's containment objective or debit spread/option buying's ATR-bounded structure do).

Requires notebook 12 to have been run with real deep history first (`from_date=2015`, `force_refresh=True`) — a real regression was found and fixed where notebook 14's live refresh was silently shrinking that history back down to ~1 year on every routine run (see README Stage 10/11). If `up_closed_count` looks thin across the board, that's the symptom — re-run notebook 12.

In [1]:
import sys
sys.path.append("..")

from src.strategy_scoring import score_for_pledged_trend_following, save_ranking

ranked, watchlist = score_for_pledged_trend_following()
print(f"Ranked: {len(ranked)}, Watchlist: {len(watchlist)}")
ranked["security_type"].value_counts()

Ranked: 96, Watchlist: 44


security_type
Equity    83
ETF       13
Name: count, dtype: int64

## Shares

In [2]:
ranked[ranked["security_type"] == "Equity"].reset_index(drop=True)

,symbol,security_type,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up
0,HDFCBANK,Equity,95.2,,,37,64.9,4.41,3.63,3.25
1,AXISBANK,Equity,91.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.40pp),,34,67.6,7.06,3.66,3.70
2,LT,Equity,88.4,,,40,65.0,5.63,2.70,3.66
3,SBIN,Equity,84.0,,,44,63.6,4.72,2.06,3.96
4,NESTLEIND,Equity,83.8,,,33,57.6,4.86,2.99,4.13
...,...,...,...,...,...,...,...,...,...,...
78,APOLLOHOSP,Equity,17.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.83pp),,43,46.5,3.61,-2.22,5.33
79,FORTIS,Equity,15.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 9.26pp),,30,46.7,8.46,-0.80,7.30
80,RAMCOCEM,Equity,13.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 7.02pp),,31,48.4,4.64,-2.38,6.30
81,SUNDRMFAST,Equity,10.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 9.49pp),⚠ HIGH GAP RISK (>10.0%),27,44.4,8.17,-1.32,6.49


## ETFs

In [3]:
ranked[ranked["security_type"] == "ETF"].reset_index(drop=True)

,symbol,security_type,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up
0,MON100,ETF,79.7,,⚠ HIGH GAP RISK (>10.0%),26,80.8,10.46,7.65,3.86
1,MAFANG,ETF,79.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.01pp),⚠ HIGH GAP RISK (>10.0%),17,76.5,10.93,5.92,2.10
2,PVTBANIETF,ETF,68.9,,⚠ HIGH GAP RISK (>10.0%),15,66.7,5.51,4.22,4.88
3,CONSUMBEES,ETF,64.7,,⚠ HIGH GAP RISK (>10.0%),18,66.7,7.17,4.77,6.48
4,NV20BEES,ETF,63.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 6.86pp),⚠ HIGH GAP RISK (>10.0%),19,84.2,9.65,2.79,3.06
5,BANKBEES,ETF,62.2,,⚠ HIGH GAP RISK (>10.0%),37,62.2,4.52,2.78,3.04
6,JUNIORBEES,ETF,50.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 7.60pp),⚠ HIGH GAP RISK (>10.0%),20,65.0,9.87,2.27,2.55
7,INFRABEES,ETF,42.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.82pp),⚠ HIGH GAP RISK (>10.0%),27,59.3,5.13,1.31,3.40
8,NIFTYBEES,ETF,40.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.75pp),⚠ HIGH GAP RISK (>10.0%),28,60.7,5.15,0.40,2.42
9,NIF100BEES,ETF,38.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.82pp),⚠ HIGH GAP RISK (>10.0%),25,56.0,5.05,1.23,2.59


In [4]:
save_ranking(ranked, "RANKING_pledged_trend_following.xlsx")
save_ranking(watchlist, "WATCHLIST_pledged_trend_following.xlsx")

Saved ranking to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/RANKING_pledged_trend_following.xlsx
Saved ranking to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/WATCHLIST_pledged_trend_following.xlsx


'/Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/WATCHLIST_pledged_trend_following.xlsx'

In [5]:
# Instruments below the 15 UP-closed-flip floor - not blended into the
# ranking, but not dropped either. Re-check as more calendar time passes.
print(f"On watchlist: {len(watchlist)}")
watchlist.sort_values("avg_pct_move_up", ascending=False).reset_index(drop=True)

On watchlist: 44


,symbol,security_type,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up
0,MNC,ETF,4,75.0,201.11,34.74,3.56
1,ANANDRATHI,Equity,12,58.3,34.20,0.43,4.12
2,GOLDCASE,ETF,3,100.0,29.03,7.00,1.48
3,LENSKART,Equity,1,100.0,18.18,18.18,3.22
4,RAINBOW,Equity,8,75.0,16.55,13.36,7.22
5,LOWVOL1,ETF,3,100.0,15.79,3.78,4.37
6,ALPL30IETF,ETF,8,62.5,14.21,4.12,2.00
7,ANURAS,Equity,14,50.0,12.63,2.42,6.36
8,MOCAPITAL,ETF,3,100.0,11.98,8.62,3.20
9,AUTOBEES,ETF,8,50.0,11.76,5.66,4.26
